In [ ]:
import os
import sys
import numpy as np
import librosa
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,Activation,Flatten
from tensorflow.keras.optimizers import Adam
from sklearn import metrics
import tensorflow.keras
import tensorflow.keras.layers as layers
import IPython.display as ipd
from tensorflow.keras.callbacks import ReduceLROnPlateau
import matplotlib.pyplot as plt
import pyswarms as ps
from tensorflow.keras.callbacks import EarlyStopping
from pyswarms.utils.search import RandomSearch
from pyswarms.single.global_best import GlobalBestPSO



print("Python version:", sys.version)
print("NumPy version:", np.__version__)
print("librosa version:", librosa.__version__)
print("TensorFlow version:", tf.__version__)
print("Pyswarms version:", ps.__version__)

Python version: 3.8.20 (default, Oct  3 2024, 15:19:54) [MSC v.1929 64 bit (AMD64)]
NumPy version: 1.19.5
librosa version: 0.9.2
TensorFlow version: 2.5.3
Pyswarms version: 1.3.0


In [ ]:
def preprocess_file(song_path, n_mfcc=128, num_segments=14):
    """
    Preprocess a single song into MFCC feature segments.
    Each segment is reshaped to (16, 8, 1).
    """
    mfcc_segments = []

    try:
        duration = librosa.get_duration(filename=song_path)

        if duration > 90:  # longer than 1.5 minutes
            audio, sr = librosa.load(
                song_path,
                sr=None,
                offset=30,
                duration=60
            )
        else:
            audio, sr = librosa.load(song_path, sr=None)

        total_samples = len(audio)
        segment_length = total_samples // num_segments

        for s in range(num_segments):
            start = s * segment_length
            end = start + segment_length
            segment = audio[start:end]

            if len(segment) < segment_length // 2:
                continue

            mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=n_mfcc)
            mfcc_mean = np.mean(mfcc.T, axis=0)
            mfcc_reshaped = np.reshape(mfcc_mean, (16, 8, 1))
            mfcc_segments.append(mfcc_reshaped)

    except Exception as e:
        print(f"Error processing {song_path}: {e}")

    return mfcc_segments


def preprocess_dataset(parent_dir, genres, n_mfcc=128, num_segments=14, max_files=None):
    dataset = []
    labels = []

    for genre, genre_number in genres.items():
        genre_dir = os.path.join(parent_dir, genre)
        print(f"Processing genre: {genre}")

        files = [f for f in os.listdir(genre_dir) if f.endswith((".wav", ".au", ".mp3"))]
        if max_files:
            files = files[:max_files]

        for filename in files:
            song_path = os.path.join(genre_dir, filename)
            mfcc_segments = preprocess_file(song_path, n_mfcc, num_segments)

            dataset.extend(mfcc_segments)
            labels.extend([genre_number] * len(mfcc_segments))

    dataset = np.array(dataset)
    labels = np.array(labels)

    print("Final dataset shape:", dataset.shape)
    print("Final labels shape:", labels.shape)
    return dataset, labels

In [ ]:
genres = {
    'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4,
    'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9
}

X, y = preprocess_dataset("data/genresWAV", genres)

Processing genre: blues
Processing genre: classical
Processing genre: country
Processing genre: disco
Processing genre: hiphop
Processing genre: jazz
Processing genre: metal
Processing genre: pop
Processing genre: reggae
Processing genre: rock
Final dataset shape: (14000, 16, 8, 1)
Final labels shape: (14000,)


In [ ]:
labelencoder = LabelEncoder()
y = to_categorical(labelencoder.fit_transform(y))


from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=0)

In [ ]:

# Define search space:
# [learning_rate, filters, dense_units, dropout]
bounds = ([1e-4, 16, 128],    # min values
          [1e-2, 64, 512])    # max values

def create_model(lr, filters, dense_units):
    model = models.Sequential([
        layers.Conv2D(int(filters), (3,3), activation='relu', padding='valid',input_shape=(16,8,1)),
        layers.MaxPooling2D(2, padding='same'),
        layers.Flatten(),
        layers.Dense(int(dense_units), activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['acc'])
    return model

def objective_function(params):
    losses = []
    for lr, filters, dense_units in params:


        model = create_model(lr, filters, dense_units)

        early_stop = EarlyStopping(
            monitor="val_loss",   # watch validation accuracy
            patience=3,               # stop if no improvement after 3 epochs
            restore_best_weights=True # rollback to best model
        )

        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=15, batch_size=128, verbose=1  # keep short for search
            ,callbacks=[early_stop]
        )

        val_acc = history.history['val_acc'][-1]
        losses.append(1 - val_acc)  # minimize (1 - accuracy)
    return np.array(losses)

In [ ]:


options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}

optimizer = GlobalBestPSO(n_particles=10, dimensions=3, options=options,
                          bounds=bounds)

# Optimize
best_cost, best_pos = optimizer.optimize(objective_function, iters=10)

print("Best params found:")
print("Learning rate:", best_pos[0])
print("Filters:", int(best_pos[1]))
print("Dense units:", int(best_pos[2]))

2026-06-26 23:59:43,468 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best:   0%|                                        |0/10

Epoch 1/15
77/77 [==============================] - 4s 13ms/step - loss: 4.1608 - acc: 0.3236 - val_loss: 1.3655 - val_acc: 0.5210
Epoch 2/15
77/77 [==============================] - 1s 10ms/step - loss: 1.2029 - acc: 0.5865 - val_loss: 1.1570 - val_acc: 0.5960
Epoch 3/15
77/77 [==============================] - 1s 9ms/step - loss: 0.9780 - acc: 0.6656 - val_loss: 0.9773 - val_acc: 0.6757
Epoch 4/15
77/77 [==============================] - 1s 9ms/step - loss: 0.8268 - acc: 0.7221 - val_loss: 0.9130 - val_acc: 0.6914
Epoch 5/15
77/77 [==============================] - 1s 9ms/step - loss: 0.7454 - acc: 0.7434 - val_loss: 0.8140 - val_acc: 0.7398
Epoch 6/15
77/77 [==============================] - 1s 9ms/step - loss: 0.6306 - acc: 0.7820 - val_loss: 0.8159 - val_acc: 0.7374
Epoch 7/15
77/77 [==============================] - 1s 10ms/step - loss: 0.5582 - acc: 0.8105 - val_loss: 0.6945 - val_acc: 0.7724
Epoch 8/15
77/77 [==============================] - 1s 12ms/step - loss: 0.5109 - acc: 

pyswarms.single.global_best:  10%|██▎                    |1/10, best_cost=0.156

Epoch 1/15
77/77 [==============================] - 2s 16ms/step - loss: 4.1153 - acc: 0.3159 - val_loss: 1.4019 - val_acc: 0.5176
Epoch 2/15
77/77 [==============================] - 1s 13ms/step - loss: 1.2030 - acc: 0.5786 - val_loss: 1.0877 - val_acc: 0.6274
Epoch 3/15
77/77 [==============================] - 1s 14ms/step - loss: 0.9845 - acc: 0.6584 - val_loss: 0.9290 - val_acc: 0.6788
Epoch 4/15
77/77 [==============================] - 1s 13ms/step - loss: 0.8504 - acc: 0.7070 - val_loss: 0.8756 - val_acc: 0.7024
Epoch 5/15
77/77 [==============================] - 1s 13ms/step - loss: 0.7264 - acc: 0.7523 - val_loss: 0.8093 - val_acc: 0.7274
Epoch 6/15
77/77 [==============================] - 1s 14ms/step - loss: 0.6754 - acc: 0.7652 - val_loss: 0.7599 - val_acc: 0.7476
Epoch 7/15
77/77 [==============================] - 1s 14ms/step - loss: 0.6428 - acc: 0.7741 - val_loss: 0.9000 - val_acc: 0.7050
Epoch 8/15
77/77 [==============================] - 1s 14ms/step - loss: 0.5724 - a

pyswarms.single.global_best:  20%|████▌                  |2/10, best_cost=0.156

...

Epoch 1/15
77/77 [==============================] - 2s 17ms/step - loss: 2.8297 - acc: 0.4026 - val_loss: 1.2641 - val_acc: 0.5764
Epoch 2/15
77/77 [==============================] - 1s 17ms/step - loss: 1.0985 - acc: 0.6248 - val_loss: 1.0153 - val_acc: 0.6588
Epoch 3/15
77/77 [==============================] - 1s 15ms/step - loss: 0.8555 - acc: 0.7074 - val_loss: 0.8975 - val_acc: 0.7019
Epoch 4/15
77/77 [==============================] - 1s 15ms/step - loss: 0.7098 - acc: 0.7553 - val_loss: 0.8124 - val_acc: 0.7295
Epoch 5/15
77/77 [==============================] - 1s 16ms/step - loss: 0.6167 - acc: 0.7862 - val_loss: 0.8021 - val_acc: 0.7288
Epoch 6/15
77/77 [==============================] - 1s 15ms/step - loss: 0.5329 - acc: 0.8126 - val_loss: 0.6825 - val_acc: 0.7676
Epoch 7/15
77/77 [==============================] - 1s 16ms/step - loss: 0.4552 - acc: 0.8442 - val_loss: 0.7811 - val_acc: 0.7540
Epoch 8/15
77/77 [==============================] - 1s 16ms/step - loss: 0.4193 - a

pyswarms.single.global_best:  90%|█████████████████████▌  |9/10, best_cost=0.13

Epoch 1/15
77/77 [==============================] - 2s 18ms/step - loss: 2.9293 - acc: 0.3821 - val_loss: 1.3253 - val_acc: 0.5379
Epoch 2/15
77/77 [==============================] - 1s 15ms/step - loss: 1.1500 - acc: 0.6090 - val_loss: 1.0601 - val_acc: 0.6331
Epoch 3/15
77/77 [==============================] - 1s 16ms/step - loss: 0.9284 - acc: 0.6784 - val_loss: 0.8737 - val_acc: 0.7007
Epoch 4/15
77/77 [==============================] - 1s 16ms/step - loss: 0.7580 - acc: 0.7370 - val_loss: 0.7969 - val_acc: 0.7312
Epoch 5/15
77/77 [==============================] - 1s 15ms/step - loss: 0.6820 - acc: 0.7644 - val_loss: 0.7968 - val_acc: 0.7367
Epoch 6/15
77/77 [==============================] - 1s 15ms/step - loss: 0.5683 - acc: 0.8044 - val_loss: 0.7229 - val_acc: 0.7486
Epoch 7/15
77/77 [==============================] - 1s 15ms/step - loss: 0.4825 - acc: 0.8349 - val_loss: 0.6209 - val_acc: 0.7971
Epoch 8/15
77/77 [==============================] - 1s 16ms/step - loss: 0.4432 - a

pyswarms.single.global_best: 100%|███████████████████████|10/10, best_cost=0.13
2026-06-27 00:26:28,082 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.130476176738739, best pos: [1.04869341e-03 5.02184859e+01 4.21956942e+02]


Best params found:
Learning rate: 0.0010486934124383427
Filters: 50
Dense units: 421
